In [11]:
import os 
from dotenv import load_dotenv

load_dotenv()


True

In [12]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH="telecom_guide.pdf"

loader=PyPDFLoader(PDF_PATH)
pages=loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n---First page preview (first 500 chars)---")
print(pages[0].page_content[:500])

Loaded 9 pages from the PDF.

---First page preview (first 500 chars)---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n","\n","."," "],
)
chunks = splitter.split_documents(pages)
len(chunks)

37

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks,embeddings)
print(f"Vector Sotre ready.{vector_store._collection.count()}vector stored")

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6866.73it/s]


Vector Sotre ready.74vector stored


In [15]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})
test_query="what is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

for i,doc in enumerate(retrieved, 1):
    print(f"---Chunk {i}---")
    print(doc.page_content[:300])
    print()
    

---Chunk 1---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

---Chunk 2---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

---Chunk 3---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt



In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
    return "\n\n----\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT="""\
You are helpfull telecom assistant.
Answer the question using ONLY the context provide below.
If the context does not contain enough information,say so clearly

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system",SYSTEM_PROMPT),
    ("human","{question}"),
])
llm = ChatGroq(
    model = "qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed",
)
chain=(
    {"context": retriever | format_docs, "question":RunnablePassthrough() }
    | prompt
    |llm
    |StrOutputParser()
)

print("RAG chain assembled.")


RAG chain assembled.


In [24]:
question="what is VoLTE.what are the benifits of that"
print(f"Question: {question}\n")
answer=chain.invoke(question)

print("Answer: ",answer)

Question: what is VoLTE.what are the benifits of that

Answer:  **VoLTE (Voice over LTE)** is an IP-based voice technology that transmits voice calls as data packets over the LTE network using the **IMS (IP Multimedia Subsystem)** core. It replaces the legacy circuit-switched voice channels used in 2G and 3G networks.  

**Key Benefits of VoLTE:**  
1. **HD Voice Quality**: Uses wideband audio at **16 kHz** (compared to 3.4 kHz in legacy calls).  
2. **Faster Call Setup**: Connects in **under 2 seconds** (vs. 6–8 seconds on 3G).  
3. **Simultaneous Voice and Data**: Allows data usage (e.g., internet browsing) during calls without degradation.  
4. **Improved Network Efficiency**: Reduces reliance on older 2G/3G networks, optimizing LTE resources.  

VoLTE requires a compatible device, a VoLTE-enabled SIM, and an account with VoLTE activated.
